# Módulo 6 — Buenas Prácticas de BigQuery: Pipeline e2e de Retention Risk

**Sesión 3 (parte 2) — Lunes 4 de Mayo, 2026**

Este notebook es el **proyecto end-to-end** que une todo lo aprendido en sesiones 1 y 2 (Bronze → Silver → Gold, ingesta, eventos) y aplica las buenas prácticas de BigQuery (M6) para construir un pipeline de producción real.

**Caso de negocio:** scoring mensual de **riesgo de retención** sobre los 209 empleados anonimizados de Personio. Cada fin de mes:
1. Construir un *feature store* particionado y clusterizado.
2. Entrenar (o re-entrenar) un modelo BQML Logistic Regression.
3. Aplicar una **regla de decisión newsvendor** (Cu vs Co) para producir una lista priorizada de empleados para HRBP.
4. Snapshot, audit log y observabilidad de coste.

**Estructura — espejo del template `04_baseline_procurement.ipynb`** (que vieron como ejemplo de notebook de Workbench en producción):

| § | Tema | Buenas prácticas M6 cubiertas |
|---|------|-------------------------------|
| 1 | Setup | Modelado de datasets de producción (T6.1) |
| 2 | Carga de las 3 tablas | Validación de capa Silver |
| 3 | Construcción del panel temporal (feature store) | Particionado + clustering (T6.2) |
| 4 | Modelo de coste: Cu, Co, q* | Regla de decisión newsvendor |
| 5 | UDFs reutilizables | T6.11 parte 1 |
| 6 | Stored Procedure idempotente | T6.11 parte 2 + control de coste (T6.3) |
| 7 | Procedural Language: backfill con WHILE | T6.11 parte 3 |
| 8 | Hold-out spec (rolling-origin) | Honestidad metodológica |
| 9 | Modelo BQML Logistic Regression | Optimización (T6.4) |
| 10 | Aplicación de la regla de decisión | SP `sp_apply_decision_rule` |
| 11 | Comparison plot — modelo vs naive base rate | Validación honesta |
| 12 | Capa semántica + authorized views + snapshots | T6.5, T6.6, T6.7, T6.8 |
| 13 | INFORMATION_SCHEMA cost audit | T6.10 |
| 14 | GenAI brief stub | Foreshadowing M16 |
| 15 | Vertex AI Model Registry (comentado) | Foreshadowing M15 |
| 16 | Findings honestos | Limitaciones, sesgos, próximos pasos |

> **Filosofía:** este notebook **no es un experimento exploratorio**. Es la pinta real que tiene un Workbench notebook en producción: idempotente, observable, con guardarraíles de coste, documentación inline y output reproducible.


---
## 1. Setup

Detección de entorno (Vertex AI Workbench vs local), clientes BQ/GCS, configuración del proyecto y creación de los datasets nuevos que necesita el pipeline:

- `feature_store_retention` — features ML versionadas, particionadas por mes
- `predictions_retention` — outputs del modelo y decisiones
- `ml_models` — modelos BQML entrenados
- `pipeline_runs` — log de ejecución y métricas de coste (observabilidad)

Esto materializa **T6.1 (Modelado de datasets de producción)**.

In [1]:
# Instalar dependencias (descomentar en primera ejecución)
# !pip install google-cloud-bigquery google-cloud-storage google-cloud-aiplatform pandas pyarrow db-dtypes python-dotenv matplotlib

import os
import json
import time
import uuid
from datetime import datetime, date, timezone
from typing import Optional
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.cloud import bigquery, storage
from google.api_core.exceptions import NotFound, Conflict, BadRequest
import warnings
warnings.filterwarnings('ignore')

# --- Detección de entorno ---
IN_VERTEX_AI = any([
    os.environ.get("DL_ANACONDA_HOME"),
    os.path.exists("/opt/deeplearning/metadata"),
    os.environ.get("GOOGLE_CLOUD_PROJECT"),
])

if IN_VERTEX_AI:
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "people-analytics-formacion")
    bq_client = bigquery.Client(project=PROJECT_ID)
    gcs_client = storage.Client(project=PROJECT_ID)
    print(f"Entorno: Vertex AI Workbench (ADC)")
else:
    from dotenv import load_dotenv
    from google.oauth2 import service_account
    load_dotenv()
    PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "people-analytics-formacion")
    creds_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS", "service-account.json")
    if os.path.exists(creds_path):
        credentials = service_account.Credentials.from_service_account_file(creds_path)
        bq_client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
        gcs_client = storage.Client(project=PROJECT_ID, credentials=credentials)
    else:
        bq_client = bigquery.Client(project=PROJECT_ID)
        gcs_client = storage.Client(project=PROJECT_ID)
    print(f"Entorno: Local")

REGION = "europe-southwest1"
BQ_LOCATION = "europe-southwest1"  # mismo que region — multi-region 'EU' también valdría

# Datasets que vamos a crear/usar
DS_BRONZE   = "bronze_personio"
DS_SILVER   = "silver_personio"
DS_GOLD     = "gold_people_analytics"
DS_FEATURES = "feature_store_retention"
DS_PREDS    = "predictions_retention"
DS_MODELS   = "ml_models"
DS_RUNS     = "pipeline_runs"

# Labels universales para FinOps (aparecerán en todas las queries críticas)
LABELS_PIPELINE = {"team": "people-analytics", "pipeline": "retention-risk", "env": "prod"}

# Guardarraíl de coste — toda query del pipeline debe respetarlo
MAX_BYTES_BILLED = 50 * 1024 ** 3  # 50 GiB por query

print(f"Proyecto:   {PROJECT_ID}")
print(f"Región:     {REGION}")
print(f"Datasets:   {DS_FEATURES}, {DS_PREDS}, {DS_MODELS}, {DS_RUNS} (a crear)")
print(f"Coste max:  {MAX_BYTES_BILLED / 1024**3:.0f} GiB / query")


Entorno: Local
Proyecto:   project-9176af0b-ecb3-4050-859
Región:     europe-southwest1
Datasets:   feature_store_retention, predictions_retention, ml_models, pipeline_runs (a crear)
Coste max:  50 GiB / query


In [2]:
# Habilitar APIs necesarias (idempotente)
import subprocess

APIS = [
    "bigquery.googleapis.com",
    "bigqueryconnection.googleapis.com",  # para BigLake si se usa
    "aiplatform.googleapis.com",          # Vertex AI (Sec. 15)
    "storage.googleapis.com",
]

for api in APIS:
    print(f"Habilitando {api} ...", end=" ")
    r = subprocess.run(["gcloud", "services", "enable", api, f"--project={PROJECT_ID}"],
                       capture_output=True, text=True)
    print("OK" if r.returncode == 0 else f"FALLO: {r.stderr.strip()[:200]}")


Habilitando bigquery.googleapis.com ... 

OK
Habilitando bigqueryconnection.googleapis.com ... 

OK
Habilitando aiplatform.googleapis.com ... 

OK
Habilitando storage.googleapis.com ... 

OK


In [3]:
# T6.1 — Crear los 4 datasets nuevos del pipeline con metadata explícita
def get_or_create_dataset(dataset_id: str, description: str, labels: dict, location: str = BQ_LOCATION):
    full_id = f"{PROJECT_ID}.{dataset_id}"
    try:
        ds = bq_client.get_dataset(full_id)
        print(f"  Dataset ya existe: {dataset_id}")
        # Actualizar metadata si hace falta
        ds.description = description
        ds.labels = labels
        bq_client.update_dataset(ds, ["description", "labels"])
    except NotFound:
        ds = bigquery.Dataset(full_id)
        ds.location = location
        ds.description = description
        ds.labels = labels
        ds.default_table_expiration_ms = None  # producción → sin expiración por defecto
        ds = bq_client.create_dataset(ds)
        print(f"  Dataset creado:    {dataset_id}")
    return ds

get_or_create_dataset(
    DS_FEATURES,
    description=("Feature store mensual del modelo de retention risk. "
                 "Granularidad: 1 fila por (employee_code, snapshot_month). "
                 "Owner: data-eng-team. Retención: 24 meses."),
    labels={**LABELS_PIPELINE, "layer": "feature_store"},
)

get_or_create_dataset(
    DS_PREDS,
    description=("Predicciones y decisiones del pipeline de retention. "
                 "Una fila por (employee_code × run_date). "
                 "Permisos: HRBP via authorized view; tabla base solo SAs ML."),
    labels={**LABELS_PIPELINE, "layer": "predictions"},
)

get_or_create_dataset(
    DS_MODELS,
    description="Modelos BQML del pipeline de people analytics. Cada modelo versionado con sufijo _vN.",
    labels={**LABELS_PIPELINE, "layer": "ml_models"},
)

get_or_create_dataset(
    DS_RUNS,
    description=("Audit log del pipeline. Cada CALL de Stored Procedure escribe una fila aquí. "
                 "Cruzar con INFORMATION_SCHEMA.JOBS_BY_PROJECT para análisis FinOps."),
    labels={**LABELS_PIPELINE, "layer": "observability"},
)

print()
print("Inventario de datasets:")
for ds in bq_client.list_datasets():
    print(f"  - {ds.dataset_id}")


  Dataset ya existe: feature_store_retention


  Dataset ya existe: predictions_retention


  Dataset ya existe: ml_models


  Dataset ya existe: pipeline_runs



Inventario de datasets:
  - bronze_personio
  - bronze_personio_dev
  - feature_store_retention
  - gold_people_analytics
  - gold_people_analytics_dev
  - ml_features
  - ml_models
  - pipeline_runs
  - predictions_retention
  - silver_personio
  - silver_personio_dev


---
## 2. Carga de las 3 tablas (validación de la capa Silver)

El pipeline depende de 2 tablas de Silver creadas en M2:

| Tabla Silver | Descripción | Fuente |
|--------------|-------------|--------|
| `silver_personio.dim_employee` | Maestro de empleados (1 fila/empleado, snapshot actual con `termination_date`, `termination_type`) | `personio_data_v2_anon.csv` |
| `silver_personio.fact_salary_history` | Snapshots mensuales con salario, fte, status — fuente principal de features | `personio_history_v2_anon.csv` |

Validamos su existencia y estructura antes de seguir. Si falta alguna, el alumno debe volver a M2.

> **Nota sobre el schema:** la capa Silver creada en M2 utiliza `fact_salary_history` como tabla de snapshots mensuales (con salario, fte, status incluidos). No existe `fact_employee_history` separada — es una decisión de modelado del cliente. Construimos las features sobre esta única tabla mensual, joineando con `dim_employee` para metadata estable (country, hire_date, termination_type del label).

In [4]:
# Validar que las tablas Silver existen y tienen filas
TABLAS_REQUERIDAS = {
    f"{DS_SILVER}.dim_employee": "Maestro de empleados",
    f"{DS_SILVER}.fact_salary_history": "Snapshots mensuales con salario y status",
}

print("Validación capa Silver:")
print(f"{'Tabla':<55} {'Filas':>10} {'Tamaño (MB)':>15}")
print("-" * 85)

todas_ok = True
for table_id, desc in TABLAS_REQUERIDAS.items():
    full_id = f"{PROJECT_ID}.{table_id}"
    try:
        t = bq_client.get_table(full_id)
        size_mb = t.num_bytes / 1024**2
        print(f"{table_id:<55} {t.num_rows:>10,} {size_mb:>15.2f}")
    except NotFound:
        print(f"{table_id:<55} {'NO EXISTE':>10}")
        todas_ok = False

if not todas_ok:
    raise RuntimeError(
        "Faltan tablas en silver_personio. Ejecuta primero el notebook de M2 "
        "para construir la capa Silver desde Bronze."
    )
print("\nOK — capa Silver lista para construir features.")


Validación capa Silver:
Tabla                                                        Filas     Tamaño (MB)
-------------------------------------------------------------------------------------


silver_personio.dim_employee                                   209            0.11


silver_personio.fact_salary_history                          1,189            0.36

OK — capa Silver lista para construir features.


In [5]:
# Distribución de la variable objetivo: ¿quién se ha dado de baja voluntariamente?
# El label vive en dim_employee (termination_type), no en fact_salary_history
sql_target_dist = f'''
SELECT
  COALESCE(termination_type, 'Activo') AS termination_type,
  COUNT(*) AS empleados,
  COUNT(termination_date) AS con_termination_date,
  ROUND(COUNT(termination_date) / COUNT(*) * 100, 1) AS pct
FROM `{PROJECT_ID}.{DS_SILVER}.dim_employee`
GROUP BY termination_type
ORDER BY empleados DESC
'''

job_config = bigquery.QueryJobConfig(
    maximum_bytes_billed=MAX_BYTES_BILLED,
    labels={**LABELS_PIPELINE, "step": "validation"},
)
df_target = bq_client.query(sql_target_dist, job_config=job_config).to_dataframe()
print("Distribución de termination_type en dim_employee:")
print(df_target.to_string(index=False))
print()
print("Decisión metodológica: el modelo predecirá únicamente 'employee-quit' (voluntary).")
print("Vocabulario normalizado en M2: 'employee-quit' = voluntario,")
print("'fired' / 'contract-expired' = involuntarios (decisión de empresa, no se 'previene').")


Distribución de termination_type en dim_employee:
termination_type  empleados  con_termination_date   pct
          active        123                     0   0.0
   employee-quit         61                    61 100.0
           fired         18                    18 100.0
contract-expired          7                     7 100.0

Decisión metodológica: el modelo predecirá únicamente 'employee-quit' (voluntary).
Vocabulario normalizado en M2: 'employee-quit' = voluntario,
'fired' / 'contract-expired' = involuntarios (decisión de empresa, no se 'previene').


---
## 3. Construcción del panel temporal — feature store particionado y clusterizado

Creamos `feature_store_retention.features`: una fila por **(employee_code × snapshot_month)** con todas las variables que alimentarán el modelo, más el label `voluntary_exit_within_3m` (1 si el empleado se da de baja voluntaria en los próximos 3 meses).

**Decisiones de modelado físico (T6.2):**
- `PARTITION BY snapshot_month` → toda query con `WHERE snapshot_month = ...` lee solo 1 partición.
- `CLUSTER BY country, team` → filtros por país/team son ~10× más rápidos.
- `require_partition_filter = TRUE` → guardarraíl: queries sin filtro de partición fallan.
- `description` y `labels` → T6.9 (documentación) y T6.10 (FinOps).

**Sobre las cohortes:** Personio en este cliente no usa bandas P1/E1/M2 explícitas — usa `team` y `position`. Lo asumimos como cohorte natural. En empresas con bandas formales se sustituiría trivialmente.

In [6]:
# Crear la tabla del feature store con todos los guardarraíles de M6
ddl_features = f'''
CREATE TABLE IF NOT EXISTS `{PROJECT_ID}.{DS_FEATURES}.features` (
  snapshot_month             DATE       NOT NULL  OPTIONS(description="Primer día del mes del snapshot"),
  employee_code              INT64      NOT NULL  OPTIONS(description="Identificador único del empleado (anonimizado)"),
  country                    STRING                OPTIONS(description="País de contrato (de dim_employee)"),
  team                       STRING                OPTIONS(description="Team del empleado en el snapshot"),
  position                   STRING                OPTIONS(description="Position del empleado en el snapshot"),
  tenure_months              INT64                 OPTIONS(description="Antigüedad en meses al snapshot"),
  tenure_bucket              STRING                OPTIONS(description="Bucket de antigüedad: 0_lt_6m, 1_6_12m, 2_1_3y, 3_3_5y, 4_gt_5y"),
  gross_salary_monthly_lc    FLOAT64               OPTIONS(description="Salario bruto mensual en local currency"),
  bonus_monthly_lc           FLOAT64               OPTIONS(description="Bonus mensual en local currency"),
  compa_ratio_team           FLOAT64               OPTIONS(description="Salario / mediana del team en mismo país y mes. <0.85 underpaid, >1.15 overpaid"),
  salary_delta_pct_6m        FLOAT64               OPTIONS(description="Variación salarial % en los últimos 6 meses"),
  has_bonus                  BOOL                  OPTIONS(description="TRUE si bonus_monthly_lc > 0 en el mes"),
  fte                        FLOAT64               OPTIONS(description="Full-time equivalent (1.0 = jornada completa)"),
  voluntary_exit_within_3m   INT64                 OPTIONS(description="Label: 1 si voluntary exit en los 3 meses siguientes, 0 si activo")
)
PARTITION BY snapshot_month
CLUSTER BY country, team
OPTIONS(
  description = "Feature store mensual para retention risk model. Granularidad: 1 fila/(empleado×mes). Refrescada por sp_build_retention_features.",
  labels = [("team", "people-analytics"), ("pipeline", "retention-risk"), ("contains_pii", "false")],
  require_partition_filter = TRUE,
  partition_expiration_days = 730
)
'''

job_config = bigquery.QueryJobConfig(
    maximum_bytes_billed=MAX_BYTES_BILLED,
    labels={**LABELS_PIPELINE, "step": "ddl_features"},
)
bq_client.query(ddl_features, job_config=job_config).result()
print(f"Tabla creada: {DS_FEATURES}.features")
print("  - Particionada por snapshot_month")
print("  - Clusterizada por country, team")
print("  - require_partition_filter = TRUE  (guardarraíl)")
print("  - 14 columnas con description individual")


Tabla creada: feature_store_retention.features
  - Particionada por snapshot_month
  - Clusterizada por country, team
  - require_partition_filter = TRUE  (guardarraíl)
  - 14 columnas con description individual


In [7]:
# Tabla de logs del pipeline (T6.10 — observabilidad)
ddl_runs = '''
CREATE TABLE IF NOT EXISTS `{project}.{ds_runs}.retention_pipeline_runs` (
  run_id           STRING     NOT NULL OPTIONS(description="UUID único del run"),
  step             STRING     NOT NULL OPTIONS(description="Nombre del paso: build_features, train_model, score, apply_decision"),
  snapshot_month   DATE                OPTIONS(description="Mes que se está procesando"),
  rows_written     INT64               OPTIONS(description="Filas escritas por el step"),
  start_time       TIMESTAMP  NOT NULL,
  end_time         TIMESTAMP,
  duration_seconds FLOAT64,
  status           STRING     NOT NULL OPTIONS(description="SUCCESS o FAILED"),
  error_message    STRING              OPTIONS(description="Stacktrace si status=FAILED, NULL si SUCCESS"),
  bytes_processed  INT64               OPTIONS(description="Bytes leídos por el step (FinOps)"),
  cost_usd_est     FLOAT64             OPTIONS(description="Coste estimado en USD (5$/TiB on-demand)")
)
PARTITION BY DATE(start_time)
CLUSTER BY step, status
OPTIONS(
  description="Audit log de cada step del pipeline. Una fila por CALL de SP. Cruzar con INFORMATION_SCHEMA para FinOps detallado.",
  labels = [("team", "people-analytics"), ("pipeline", "retention-risk"), ("layer", "observability")]
)
'''.format(project=PROJECT_ID, ds_runs=DS_RUNS)

bq_client.query(ddl_runs).result()
print(f"Tabla creada: {DS_RUNS}.retention_pipeline_runs")


Tabla creada: pipeline_runs.retention_pipeline_runs


---
## 4. Modelo de coste — Cu, Co, q* (regla newsvendor adaptada a retención)

Este es el corazón del proyecto: **un modelo de retención no produce predicciones, produce decisiones**. La pregunta no es "¿cuál es la probabilidad de que se vaya?" sino "¿debo activar una acción de retención (que cuesta Co) sabiendo que el empleado podría irse (cuesta Cu)?".

Espejamos el cálculo del template *procurement* pero con costes de RR.HH.:

- **Cu (under-action)** = coste de no actuar y que el empleado se vaya = **coste de reemplazo**
  - Estudios consistentes: 50%–200% del salario anual según nivel.
  - Usamos **100%** del salario anual como proxy conservador.
- **Co (over-action)** = coste de actuar sobre alguien que no se iba = **bonus / coaching innecesario**
  - Bonus de retención típico: 5%–15% del salario anual.
  - Usamos **10%** del salario anual.
- **Threshold óptimo:** τ* = Cu / (Cu + Co) ≈ 0.91

Solo activamos acción de retención cuando `score ≥ 0.91` → top-decil. Todo score por debajo no compensa el coste de la acción.

In [8]:
# Parámetros de coste por nivel jerárquico (derivado de position keywords)
# Estos parámetros se ajustarían con el equipo de Compensation; aquí valores por defecto razonables.
# Tier inferido del position string: "Senior"/"Lead"/"Manager"/"Director" → tiers más altos.
COST_PARAMS = pd.DataFrame([
    {"tier": "junior",   "Cu_pct": 0.80, "Co_pct": 0.08},
    {"tier": "standard", "Cu_pct": 1.00, "Co_pct": 0.10},
    {"tier": "senior",   "Cu_pct": 1.50, "Co_pct": 0.12},
    {"tier": "lead",     "Cu_pct": 1.80, "Co_pct": 0.12},
    {"tier": "manager",  "Cu_pct": 2.00, "Co_pct": 0.15},
])
COST_PARAMS["tau_star"] = COST_PARAMS["Cu_pct"] / (COST_PARAMS["Cu_pct"] + COST_PARAMS["Co_pct"])

print("Parámetros de coste por tier:")
print(COST_PARAMS.to_string(index=False))
print()
print("Interpretación de τ*:")
print("  - Por encima de τ* → acción de retención.")
print("  - Por debajo de τ* → no actuar (el coste esperado de actuar supera el coste esperado de no actuar).")
print()
print("τ* alto (manager = 0.93) → solo actuamos si estamos muy seguros (acción cara relativa al riesgo).")
print("τ* bajo (junior = 0.91) → similar, porque para todos los tiers Co << Cu.")


Parámetros de coste por tier:
    tier  Cu_pct  Co_pct  tau_star
  junior     0.8    0.08  0.909091
standard     1.0    0.10  0.909091
  senior     1.5    0.12  0.925926
    lead     1.8    0.12  0.937500
 manager     2.0    0.15  0.930233

Interpretación de τ*:
  - Por encima de τ* → acción de retención.
  - Por debajo de τ* → no actuar (el coste esperado de actuar supera el coste esperado de no actuar).

τ* alto (manager = 0.93) → solo actuamos si estamos muy seguros (acción cara relativa al riesgo).
τ* bajo (junior = 0.91) → similar, porque para todos los tiers Co << Cu.


In [9]:
# Subimos COST_PARAMS a BQ como tabla de configuración
table_id = f"{PROJECT_ID}.{DS_GOLD}.cost_params_retention"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",
    schema=[
        bigquery.SchemaField("tier", "STRING"),
        bigquery.SchemaField("Cu_pct", "FLOAT64"),
        bigquery.SchemaField("Co_pct", "FLOAT64"),
        bigquery.SchemaField("tau_star", "FLOAT64"),
    ],
    labels={**LABELS_PIPELINE, "step": "config"},
)
bq_client.load_table_from_dataframe(COST_PARAMS, table_id, job_config=job_config).result()

# Documentación de la tabla
bq_client.query(f'''
ALTER TABLE `{table_id}`
SET OPTIONS(
  description="Parámetros de coste de la regla de decisión newsvendor por tier jerárquico. Cu_pct=coste reemplazo (% salario), Co_pct=coste retención (% salario), tau_star=threshold óptimo. Revisión semestral con Compensation."
)
''').result()
print(f"Tabla cargada: {table_id}")


Tabla cargada: project-9176af0b-ecb3-4050-859.gold_people_analytics.cost_params_retention


---
## 5. UDFs reutilizables (T6.11 parte 1)

Encapsulamos 3 reglas de negocio en UDFs SQL. Beneficios:
- **Reutilización:** las usaremos en el SP, en vistas semánticas y en queries ad-hoc.
- **Testeable:** se prueban independientemente.
- **Versionable:** un cambio en `compa_ratio` se hace en un sitio y se propaga.

In [10]:
# UDF 1: tenure_bucket — categorización de antigüedad
# UDF 2: salary_delta_pct — variación salarial entre dos valores
# UDF 3: tier_from_position — infiere tier jerárquico del nombre de la position

udfs = [
('tenure_bucket', f'''
CREATE OR REPLACE FUNCTION `{PROJECT_ID}.{DS_GOLD}.tenure_bucket`(
  hire_date DATE,
  snapshot_date DATE
) RETURNS STRING
OPTIONS(description="Categorización de antigüedad. Buckets: 0_lt_6m, 1_6_12m, 2_1_3y, 3_3_5y, 4_gt_5y, NULL si hire_date es NULL.")
AS (
  CASE
    WHEN hire_date IS NULL THEN NULL
    WHEN DATE_DIFF(snapshot_date, hire_date, MONTH) < 6 THEN '0_lt_6m'
    WHEN DATE_DIFF(snapshot_date, hire_date, MONTH) < 12 THEN '1_6_12m'
    WHEN DATE_DIFF(snapshot_date, hire_date, MONTH) < 36 THEN '2_1_3y'
    WHEN DATE_DIFF(snapshot_date, hire_date, MONTH) < 60 THEN '3_3_5y'
    ELSE '4_gt_5y'
  END
)
'''),

('salary_delta_pct', f'''
CREATE OR REPLACE FUNCTION `{PROJECT_ID}.{DS_GOLD}.salary_delta_pct`(
  current_salary FLOAT64,
  prior_salary FLOAT64
) RETURNS FLOAT64
OPTIONS(description="Variación porcentual entre dos salarios. (current - prior) / prior. NULL si prior=0 o NULL.")
AS (
  SAFE_DIVIDE(current_salary - prior_salary, prior_salary)
)
'''),

('tier_from_position', f'''
CREATE OR REPLACE FUNCTION `{PROJECT_ID}.{DS_GOLD}.tier_from_position`(
  position STRING
) RETURNS STRING
OPTIONS(description="Infiere tier jerárquico del nombre de la position (junior/standard/senior/lead/manager).")
AS (
  CASE
    WHEN position IS NULL THEN 'standard'
    WHEN REGEXP_CONTAINS(LOWER(position), r'\\b(director|head|vp|vice president|chief)\\b') THEN 'manager'
    WHEN REGEXP_CONTAINS(LOWER(position), r'\\b(manager|principal)\\b') THEN 'manager'
    WHEN REGEXP_CONTAINS(LOWER(position), r'\\b(lead|staff)\\b') THEN 'lead'
    WHEN REGEXP_CONTAINS(LOWER(position), r'\\b(senior|sr\\.?|sr )') THEN 'senior'
    WHEN REGEXP_CONTAINS(LOWER(position), r'\\b(junior|jr\\.?|intern|trainee|associate)\\b') THEN 'junior'
    ELSE 'standard'
  END
)
'''),
]

for name, ddl in udfs:
    bq_client.query(ddl).result()
    print(f"UDF creada: {DS_GOLD}.{name}")

# Test rápido de las UDFs
df_udf_test = bq_client.query(f'''
SELECT
  '{PROJECT_ID}' AS project,
  `{PROJECT_ID}.{DS_GOLD}.tenure_bucket`(DATE '2023-01-15', DATE '2025-11-01') AS tenure_test,
  `{PROJECT_ID}.{DS_GOLD}.salary_delta_pct`(55000, 50000) AS salary_delta_test,
  `{PROJECT_ID}.{DS_GOLD}.tier_from_position`('Senior Talent Acquisition Specialist') AS tier_test_1,
  `{PROJECT_ID}.{DS_GOLD}.tier_from_position`('Office Manager') AS tier_test_2,
  `{PROJECT_ID}.{DS_GOLD}.tier_from_position`('Junior Developer') AS tier_test_3
''').to_dataframe()
print()
print("Test de UDFs:")
print(df_udf_test.T.to_string(header=False))


UDF creada: gold_people_analytics.tenure_bucket


UDF creada: gold_people_analytics.salary_delta_pct


UDF creada: gold_people_analytics.tier_from_position



Test de UDFs:
project            project-9176af0b-ecb3-4050-859
tenure_test                                2_1_3y
salary_delta_test                             0.1
tier_test_1                                senior
tier_test_2                               manager
tier_test_3                                junior


---
## 6. Stored Procedure idempotente: `sp_build_retention_features` (T6.11 parte 2)

Esta es la **unidad fundamental del pipeline**. El SP:
1. Borra la partición del mes (idempotencia — re-ejecuciones seguras).
2. Reconstruye las features uniendo `dim_employee`, `fact_employee_history` y `fact_payroll_monthly`.
3. Calcula el label mirando 3 meses hacia delante en `fact_employee_history`.
4. Escribe en `feature_store_retention.features`.
5. Loguea el resultado en `pipeline_runs.retention_pipeline_runs` (success o failure).

**Buenas prácticas embebidas:**
- BEGIN ... EXCEPTION WHEN ERROR ... RAISE → captura de errores con log.
- DECLARE para variables intermedias.
- @@row_count para registrar volumen escrito.
- Filtros de partición explícitos en cada FROM (T6.3 control de coste).

In [11]:
# T6.11 — Stored Procedure idempotente con manejo de errores
sp_build_features = f'''
CREATE OR REPLACE PROCEDURE `{PROJECT_ID}.{DS_FEATURES}.sp_build_retention_features`(
  IN target_month DATE
)
OPTIONS(
  description="Construye features de retention para un mes dado. Idempotente (DELETE+INSERT). Loguea a pipeline_runs.",
  strict_mode=false
)
BEGIN
  DECLARE rows_written INT64 DEFAULT 0;
  DECLARE start_ts TIMESTAMP DEFAULT CURRENT_TIMESTAMP();
  DECLARE run_uuid STRING DEFAULT GENERATE_UUID();

  BEGIN

    -- Idempotencia
    DELETE FROM `{PROJECT_ID}.{DS_FEATURES}.features`
    WHERE snapshot_month = target_month;

    -- Construcción del panel
    INSERT INTO `{PROJECT_ID}.{DS_FEATURES}.features`
    WITH
    -- Snapshot del mes target: solo activos
    snapshot_now AS (
      SELECT
        employee_code, hire_date, team, position, status,
        gross_salary_monthly_lc, bonus_monthly_lc, fte
      FROM `{PROJECT_ID}.{DS_SILVER}.fact_salary_history`
      WHERE snapshot_date = target_month
        AND status = 'active'
    ),
    -- Salario hace 6 meses para calcular delta
    snapshot_6m_ago AS (
      SELECT employee_code, gross_salary_monthly_lc AS prior_salary
      FROM `{PROJECT_ID}.{DS_SILVER}.fact_salary_history`
      WHERE snapshot_date = DATE_SUB(target_month, INTERVAL 6 MONTH)
    ),
    -- Joineamos con dim_employee para obtener country
    snapshot_with_country AS (
      SELECT s.*, e.country
      FROM snapshot_now s
      LEFT JOIN `{PROJECT_ID}.{DS_SILVER}.dim_employee` e USING(employee_code)
    ),
    -- Mediana de salario por (country, team) en el mes target
    team_medians AS (
      SELECT
        country, team,
        APPROX_QUANTILES(gross_salary_monthly_lc, 100)[OFFSET(50)] AS median_salary_team
      FROM snapshot_with_country
      WHERE gross_salary_monthly_lc IS NOT NULL AND gross_salary_monthly_lc > 0
      GROUP BY country, team
    ),
    -- Label: ¿voluntary exit (employee-quit) en los próximos 3 meses según dim_employee?
    -- Vocabulario del cliente (normalizado en M2): 'employee-quit' = voluntary, 'fired'/'contract-expired' = involuntary
    future_voluntary_exits AS (
      SELECT employee_code, 1 AS will_exit
      FROM `{PROJECT_ID}.{DS_SILVER}.dim_employee`
      WHERE termination_type = 'employee-quit'
        AND termination_date IS NOT NULL
        AND termination_date > target_month
        AND termination_date <= DATE_ADD(target_month, INTERVAL 3 MONTH)
    )
    SELECT
      target_month AS snapshot_month,
      s.employee_code,
      s.country,
      s.team,
      s.position,
      DATE_DIFF(target_month, s.hire_date, MONTH) AS tenure_months,
      `{PROJECT_ID}.{DS_GOLD}.tenure_bucket`(s.hire_date, target_month) AS tenure_bucket,
      s.gross_salary_monthly_lc,
      s.bonus_monthly_lc,
      SAFE_DIVIDE(s.gross_salary_monthly_lc, tm.median_salary_team) AS compa_ratio_team,
      `{PROJECT_ID}.{DS_GOLD}.salary_delta_pct`(s.gross_salary_monthly_lc, s6.prior_salary) AS salary_delta_pct_6m,
      (COALESCE(s.bonus_monthly_lc, 0) > 0) AS has_bonus,
      s.fte,
      COALESCE(f.will_exit, 0) AS voluntary_exit_within_3m
    FROM snapshot_with_country s
    LEFT JOIN team_medians tm USING(country, team)
    LEFT JOIN snapshot_6m_ago s6 USING(employee_code)
    LEFT JOIN future_voluntary_exits f USING(employee_code);

    SET rows_written = @@row_count;

    -- Log SUCCESS
    INSERT INTO `{PROJECT_ID}.{DS_RUNS}.retention_pipeline_runs`
      (run_id, step, snapshot_month, rows_written, start_time, end_time, duration_seconds, status, error_message, bytes_processed, cost_usd_est)
    VALUES
      (run_uuid, 'build_features', target_month, rows_written, start_ts, CURRENT_TIMESTAMP(),
       TIMESTAMP_DIFF(CURRENT_TIMESTAMP(), start_ts, MILLISECOND) / 1000.0,
       'SUCCESS', NULL, NULL, NULL);

  EXCEPTION WHEN ERROR THEN
    INSERT INTO `{PROJECT_ID}.{DS_RUNS}.retention_pipeline_runs`
      (run_id, step, snapshot_month, rows_written, start_time, end_time, duration_seconds, status, error_message, bytes_processed, cost_usd_est)
    VALUES
      (run_uuid, 'build_features', target_month, 0, start_ts, CURRENT_TIMESTAMP(),
       TIMESTAMP_DIFF(CURRENT_TIMESTAMP(), start_ts, MILLISECOND) / 1000.0,
       'FAILED', @@error.message, NULL, NULL);
    RAISE USING MESSAGE = @@error.message;
  END;
END
'''

bq_client.query(sp_build_features).result()
print(f"SP creado: {DS_FEATURES}.sp_build_retention_features")


SP creado: feature_store_retention.sp_build_retention_features


In [12]:
# Probar el SP con un mes individual — y dry-run primero (T6.3)
TEST_MONTH = "2025-09-01"

# Dry run para ver cuánto leería
dry_sql = f"CALL `{PROJECT_ID}.{DS_FEATURES}.sp_build_retention_features`(DATE '{TEST_MONTH}')"
# (Nota: dry run de procedimientos almacenados funciona pero estima 0 — los CALL no exponen bytes)

# Ejecución real
job_config = bigquery.QueryJobConfig(
    maximum_bytes_billed=MAX_BYTES_BILLED,
    labels={**LABELS_PIPELINE, "step": "build_features", "snapshot": TEST_MONTH},
)
job = bq_client.query(dry_sql, job_config=job_config)
job.result()

print(f"SP ejecutado para snapshot_month = {TEST_MONTH}")
print(f"Bytes facturados: {job.total_bytes_billed / 1024**2:.2f} MB")
print(f"Slot-ms:          {job.slot_millis}")
print()

# Verificar que el run quedó logueado
df_run = bq_client.query(f'''
SELECT step, snapshot_month, rows_written, status, duration_seconds
FROM `{PROJECT_ID}.{DS_RUNS}.retention_pipeline_runs`
WHERE DATE(start_time) = CURRENT_DATE()
ORDER BY start_time DESC
LIMIT 5
''').to_dataframe()
print("Últimos runs en pipeline_runs:")
print(df_run.to_string(index=False))


SP ejecutado para snapshot_month = 2025-09-01
Bytes facturados: 30.00 MB
Slot-ms:          17916



Últimos runs en pipeline_runs:
            step snapshot_month  rows_written  status  duration_seconds
  build_features     2025-09-01           103 SUCCESS             4.864
           score     2025-11-01           109 SUCCESS             4.636
           score     2025-10-01           104 SUCCESS             4.561
           score     2025-09-01           103 SUCCESS             4.794
backfill_summary     2025-12-01            12 SUCCESS               NaN


---
## 7. Procedural Language: backfill con WHILE (T6.11 parte 3)

Para entrenar y validar el modelo necesitamos features de **varios meses**. En lugar de llamar al SP 12 veces desde Python, lo hacemos **dentro de BigQuery** con un bucle `WHILE` (Procedural Language).

Ventajas:
- Una sola transacción de orquestación (más fácil de monitorizar).
- El estado vive en BQ, no en el cliente Python.
- Reusable desde Cloud Workflows / Composer (M5).

In [13]:
# SP de backfill — bucle WHILE en procedural language
sp_backfill = f'''
CREATE OR REPLACE PROCEDURE `{PROJECT_ID}.{DS_FEATURES}.sp_backfill_features`(
  IN start_month DATE,
  IN end_month DATE
)
OPTIONS(
  description="Backfill iterativo del feature store. Llama a sp_build_retention_features mes a mes con WHILE loop.",
  strict_mode=false
)
BEGIN
  DECLARE current_month DATE DEFAULT start_month;
  DECLARE meses_procesados INT64 DEFAULT 0;

  WHILE current_month <= end_month DO
    CALL `{PROJECT_ID}.{DS_FEATURES}.sp_build_retention_features`(current_month);
    SET current_month = DATE_ADD(current_month, INTERVAL 1 MONTH);
    SET meses_procesados = meses_procesados + 1;
  END WHILE;

  -- Log resumen del backfill
  INSERT INTO `{PROJECT_ID}.{DS_RUNS}.retention_pipeline_runs`
    (run_id, step, snapshot_month, rows_written, start_time, end_time, status)
  VALUES
    (GENERATE_UUID(), 'backfill_summary', end_month, meses_procesados, CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP(), 'SUCCESS');
END
'''

bq_client.query(sp_backfill).result()
print(f"SP creado: {DS_FEATURES}.sp_backfill_features")


SP creado: feature_store_retention.sp_backfill_features


In [14]:
# Ejecutar backfill: 12 meses (2025-01 a 2025-12)
# Esto generará features para todo 2025 → necesario para train/test rolling-origin
backfill_call = f'''
CALL `{PROJECT_ID}.{DS_FEATURES}.sp_backfill_features`(DATE '2025-01-01', DATE '2025-12-01')
'''

job_config = bigquery.QueryJobConfig(
    maximum_bytes_billed=MAX_BYTES_BILLED * 12,  # 12 meses
    labels={**LABELS_PIPELINE, "step": "backfill"},
)
print("Ejecutando backfill 12 meses... (puede tardar 30-60s)")
job = bq_client.query(backfill_call, job_config=job_config)
job.result()
print(f"Backfill completado. Bytes facturados totales: {job.total_bytes_billed / 1024**2:.2f} MB")


Ejecutando backfill 12 meses... (puede tardar 30-60s)


Backfill completado. Bytes facturados totales: 360.00 MB


In [15]:
# Auditoría del feature store: ¿cuántas filas por mes? ¿cuál es el balance de clases?
df_audit = bq_client.query(f'''
SELECT
  snapshot_month,
  COUNT(*) AS filas,
  COUNTIF(voluntary_exit_within_3m = 1) AS positivos,
  ROUND(COUNTIF(voluntary_exit_within_3m = 1) / COUNT(*) * 100, 2) AS pct_positivos,
  COUNT(DISTINCT country) AS paises,
  COUNT(DISTINCT team) AS teams
FROM `{PROJECT_ID}.{DS_FEATURES}.features`
WHERE snapshot_month BETWEEN '2025-01-01' AND '2025-12-01'
GROUP BY snapshot_month
ORDER BY snapshot_month
''', job_config=bigquery.QueryJobConfig(labels={**LABELS_PIPELINE, "step": "audit"})).to_dataframe()

print("Auditoría del feature store por mes:")
print(df_audit.to_string(index=False))
print()
print(f"Tasa de positivos media: {df_audit['pct_positivos'].mean():.1f}%")
print()
print("HONEST READ: el dataset Personio anonimizado tiene ~41% de empleados con termination_date.")
print("Eso es una tasa enorme — probablemente la empresa pasó por un downsizing o el export incluye")
print("histórico de leavers junto con activos. Es una bandera roja a discutir con el cliente antes de ")
print("desplegar el modelo en producción.")


Auditoría del feature store por mes:
snapshot_month  filas  positivos  pct_positivos  paises  teams
    2025-01-01     87          2           2.30       8      5
    2025-02-01     88          1           1.14       8      5
    2025-03-01     88          1           1.14       8      5
    2025-04-01     90          3           3.33       8      5
    2025-05-01     91          2           2.20       8      5
    2025-06-01     93          2           2.15       8      5
    2025-07-01     99          4           4.04       8      5
    2025-08-01     99          8           8.08       8      5
    2025-09-01    103          5           4.85       8      5
    2025-10-01    104          2           1.92       8      5
    2025-11-01    109          1           0.92       9      5
    2025-12-01    109          1           0.92       9      5

Tasa de positivos media: 2.7%

HONEST READ: el dataset Personio anonimizado tiene ~41% de empleados con termination_date.
Eso es una tasa enorm

---
## 8. Hold-out specification (rolling-origin)

Para evaluar el modelo de forma honesta, **nunca usamos datos del futuro para predecir el pasado**. Hacemos una validación tipo *rolling-origin* (espejo del template procurement):

- **Train:** snapshots 2025-01 a 2025-08 (8 meses)
- **Test:** snapshots 2025-09 a 2025-11 (3 meses, con label observable porque tenemos datos hasta 2025-12)

El hold-out es **temporal**, no aleatorio — porque en producción siempre predecimos sobre meses futuros. Un split aleatorio sería *data leakage* (el modelo vería información del futuro durante el entrenamiento).

In [16]:
# Definir splits explícitamente
TRAIN_MONTHS = pd.date_range('2025-01-01', '2025-08-01', freq='MS').strftime('%Y-%m-%d').tolist()
TEST_MONTHS  = pd.date_range('2025-09-01', '2025-11-01', freq='MS').strftime('%Y-%m-%d').tolist()

print("HOLD-OUT TEMPORAL")
print(f"Train ({len(TRAIN_MONTHS)} meses): {TRAIN_MONTHS}")
print(f"Test  ({len(TEST_MONTHS)} meses): {TEST_MONTHS}")
print()
print("Nota: el snapshot 2025-12 no se usa para test porque no podemos observar el label")
print("(necesitaríamos datos hasta 2026-03 para saber si hubo voluntary exit en los 3 meses siguientes).")


HOLD-OUT TEMPORAL
Train (8 meses): ['2025-01-01', '2025-02-01', '2025-03-01', '2025-04-01', '2025-05-01', '2025-06-01', '2025-07-01', '2025-08-01']
Test  (3 meses): ['2025-09-01', '2025-10-01', '2025-11-01']

Nota: el snapshot 2025-12 no se usa para test porque no podemos observar el label
(necesitaríamos datos hasta 2026-03 para saber si hubo voluntary exit en los 3 meses siguientes).


---
## 9. Modelo BQML Logistic Regression

Entrenamos un modelo logistic regression dentro de BigQuery (sin mover datos a Vertex AI todavía — eso vendrá en M15). Es el modelo más simple razonable para un baseline honesto.

**Por qué logistic regression como baseline:**
- Interpretabilidad: cada coeficiente es leíble.
- Coste cero de infraestructura: vive en BQ.
- Si un modelo más complejo (XGBoost, redes) no supera al logistic, no merece la pena la complejidad operacional.
- Es el "smell test" estándar en ML productivo.

In [17]:
# Entrenar modelo logistic regression en BQML
# CREATE_REPLACE permite re-ejecutar el notebook sin errores
train_months_str = ", ".join(f"DATE '{m}'" for m in TRAIN_MONTHS)

train_model_sql = f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DS_MODELS}.retention_logistic_v1`
OPTIONS(
  model_type = 'LOGISTIC_REG',
  input_label_cols = ['voluntary_exit_within_3m'],
  auto_class_weights = TRUE,        -- maneja el desbalanceo si lo hay
  data_split_method = 'NO_SPLIT',   -- nosotros ya hicimos el split temporal
  enable_global_explain = TRUE,     -- para feature importance
  l2_reg = 0.1
) AS
SELECT
  -- Features (excluir employee_code que es ID, snapshot_month que es info temporal)
  country,
  team,
  tenure_months,
  tenure_bucket,
  compa_ratio_team,
  salary_delta_pct_6m,
  has_bonus,
  fte,
  voluntary_exit_within_3m
FROM `{PROJECT_ID}.{DS_FEATURES}.features`
WHERE snapshot_month IN ({train_months_str})
'''

print("Entrenando modelo BQML Logistic Regression...")
job_config = bigquery.QueryJobConfig(
    maximum_bytes_billed=MAX_BYTES_BILLED,
    labels={**LABELS_PIPELINE, "step": "train_model"},
)
job = bq_client.query(train_model_sql, job_config=job_config)
job.result()
print(f"Modelo entrenado: {DS_MODELS}.retention_logistic_v1")
print(f"Bytes facturados: {job.total_bytes_billed / 1024**2:.2f} MB")


Entrenando modelo BQML Logistic Regression...


Modelo entrenado: ml_models.retention_logistic_v1
Bytes facturados: 10.00 MB


In [18]:
# Evaluar el modelo en el TEST set (rolling-origin honest evaluation)
test_months_str = ", ".join(f"DATE '{m}'" for m in TEST_MONTHS)

eval_sql = f'''
SELECT
  *
FROM ML.EVALUATE(
  MODEL `{PROJECT_ID}.{DS_MODELS}.retention_logistic_v1`,
  (
    SELECT
      country, team, tenure_months, tenure_bucket, compa_ratio_team,
      salary_delta_pct_6m, has_bonus, fte,
      voluntary_exit_within_3m
    FROM `{PROJECT_ID}.{DS_FEATURES}.features`
    WHERE snapshot_month IN ({test_months_str})
  )
)
'''

df_eval = bq_client.query(eval_sql, job_config=bigquery.QueryJobConfig(labels={**LABELS_PIPELINE, "step": "evaluate"})).to_dataframe()
print("Métricas en TEST (3 meses, rolling-origin):")
print(df_eval.T.to_string(header=False))
print()
print("Lectura honesta:")
print(f"  - AUC ROC: {df_eval['roc_auc'].iloc[0]:.3f}  (0.5 = random, 1.0 = perfecto)")
print(f"  - Accuracy: {df_eval['accuracy'].iloc[0]:.3f}  (cuidado — puede engañar con clases desbalanceadas)")
print(f"  - Precision: {df_eval['precision'].iloc[0]:.3f}")
print(f"  - Recall: {df_eval['recall'].iloc[0]:.3f}")


Métricas en TEST (3 meses, rolling-origin):
precision  0.035714
recall     0.375000
accuracy   0.727848
f1_score   0.065217
log_loss   0.513756
roc_auc    0.734994

Lectura honesta:
  - AUC ROC: 0.735  (0.5 = random, 1.0 = perfecto)
  - Accuracy: 0.728  (cuidado — puede engañar con clases desbalanceadas)
  - Precision: 0.036
  - Recall: 0.375


In [19]:
# Crear tabla operativa de scores (lo que el Workflow escribe cada mes)
ddl_scores = f'''
CREATE TABLE IF NOT EXISTS `{PROJECT_ID}.{DS_PREDS}.scores` (
  snapshot_month DATE NOT NULL,
  employee_code INT64 NOT NULL,
  country STRING,
  team STRING,
  position STRING,
  actual_label INT64 OPTIONS(description="Label observable solo cuando snapshot_month está en hold-out histórico"),
  prob_class_0 FLOAT64,
  prob_class_1 FLOAT64,
  scored_at TIMESTAMP
)
PARTITION BY snapshot_month
CLUSTER BY country, team
OPTIONS(
  description="Scores del modelo de retention. Una fila por (employee × snapshot). Escrita por sp_score_retention cada mes.",
  labels = [("team", "people-analytics"), ("pipeline", "retention-risk")]
)
'''
bq_client.query(ddl_scores).result()

# SP de scoring — el step intermedio del Workflow
sp_score = f'''
CREATE OR REPLACE PROCEDURE `{PROJECT_ID}.{DS_PREDS}.sp_score_retention`(
  IN target_month DATE
)
OPTIONS(
  description="Aplica el modelo BQML al feature store del mes target. Idempotente.",
  strict_mode=false
)
BEGIN
  DECLARE start_ts TIMESTAMP DEFAULT CURRENT_TIMESTAMP();
  DECLARE rows_written INT64;
  DECLARE run_uuid STRING DEFAULT GENERATE_UUID();

  BEGIN
    DELETE FROM `{PROJECT_ID}.{DS_PREDS}.scores` WHERE snapshot_month = target_month;

    INSERT INTO `{PROJECT_ID}.{DS_PREDS}.scores`
    SELECT
      snapshot_month,
      employee_code,
      country,
      team,
      position,
      voluntary_exit_within_3m AS actual_label,
      predicted_voluntary_exit_within_3m_probs[OFFSET(0)].prob AS prob_class_0,
      predicted_voluntary_exit_within_3m_probs[OFFSET(1)].prob AS prob_class_1,
      CURRENT_TIMESTAMP() AS scored_at
    FROM ML.PREDICT(
      MODEL `{PROJECT_ID}.{DS_MODELS}.retention_logistic_v1`,
      (SELECT * FROM `{PROJECT_ID}.{DS_FEATURES}.features` WHERE snapshot_month = target_month)
    );

    SET rows_written = @@row_count;
    INSERT INTO `{PROJECT_ID}.{DS_RUNS}.retention_pipeline_runs`
      (run_id, step, snapshot_month, rows_written, start_time, end_time, duration_seconds, status)
    VALUES
      (run_uuid, 'score', target_month, rows_written, start_ts, CURRENT_TIMESTAMP(),
       TIMESTAMP_DIFF(CURRENT_TIMESTAMP(), start_ts, MILLISECOND) / 1000.0, 'SUCCESS');
  EXCEPTION WHEN ERROR THEN
    INSERT INTO `{PROJECT_ID}.{DS_RUNS}.retention_pipeline_runs`
      (run_id, step, snapshot_month, rows_written, start_time, end_time, status, error_message)
    VALUES
      (run_uuid, 'score', target_month, 0, start_ts, CURRENT_TIMESTAMP(), 'FAILED', @@error.message);
    RAISE USING MESSAGE = @@error.message;
  END;
END
'''
bq_client.query(sp_score).result()
print(f"SP creado: {DS_PREDS}.sp_score_retention")

# Llamar al SP para los meses de test → poblar scores
for m in TEST_MONTHS:
    bq_client.query(
        f"CALL `{PROJECT_ID}.{DS_PREDS}.sp_score_retention`(DATE '{m}')",
        job_config=bigquery.QueryJobConfig(labels={**LABELS_PIPELINE, "step": "score"})
    ).result()
    print(f"  Scores generados para {m}")

# Mantenemos scores_test como vista sobre scores filtrada (para no romper la sección 11)
bq_client.query(f'''
CREATE OR REPLACE VIEW `{PROJECT_ID}.{DS_PREDS}.scores_test` AS
SELECT * FROM `{PROJECT_ID}.{DS_PREDS}.scores`
WHERE snapshot_month IN ({test_months_str})
''').result()
print(f"Vista creada: {DS_PREDS}.scores_test (alias de scores filtrado al hold-out)")

df_scores = bq_client.query(f'''
SELECT * FROM `{PROJECT_ID}.{DS_PREDS}.scores_test`
WHERE snapshot_month = DATE '2025-11-01'
ORDER BY prob_class_1 DESC
LIMIT 10
''').to_dataframe()
print("Top 10 scores más altos en snapshot 2025-11:")
print(df_scores[['employee_code', 'country', 'team', 'actual_label', 'prob_class_1']].to_string(index=False))


SP creado: predictions_retention.sp_score_retention


  Scores generados para 2025-09-01


  Scores generados para 2025-10-01


  Scores generados para 2025-11-01


BadRequest: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/project-9176af0b-ecb3-4050-859/queries/7a1cf287-f15c-4903-8362-11ebdccf079b?maxResults=0&location=europe-southwest1&prettyPrint=false: project-9176af0b-ecb3-4050-859:predictions_retention.scores_test is not allowed for this operation because it is currently a TABLE.

Location: europe-southwest1
Job ID: 7a1cf287-f15c-4903-8362-11ebdccf079b


---
## 10. Aplicación de la regla de decisión newsvendor

Tomamos los scores y aplicamos el threshold τ* **por banda** (cada banda tiene su propio τ* según Cu/Co). Para cada empleado producimos:
- `recommended_action` ∈ {`retention_action`, `monitor`, `no_action`}
- `priority` (1-5, 1 = máxima)
- `expected_cost_no_action` = score × Cu × salario
- `expected_cost_action` = Co × salario (siempre que hagamos acción)

Esta es la **decisión real** que llega a HRBP — los scores son input intermedio.

In [ ]:
# SP que aplica la regla de decisión y produce la lista de acciones
sp_apply_decision = f'''
CREATE OR REPLACE PROCEDURE `{PROJECT_ID}.{DS_PREDS}.sp_apply_decision_rule`(
  IN target_month DATE
)
OPTIONS(
  description="Aplica regla de decisión newsvendor a los scores. Produce lista priorizada para HRBP.",
  strict_mode=false
)
BEGIN
  DECLARE start_ts TIMESTAMP DEFAULT CURRENT_TIMESTAMP();
  DECLARE rows_written INT64;
  DECLARE run_uuid STRING DEFAULT GENERATE_UUID();

  BEGIN

    -- Idempotencia
    DELETE FROM `{PROJECT_ID}.{DS_PREDS}.retention_actions`
    WHERE snapshot_month = target_month;

    INSERT INTO `{PROJECT_ID}.{DS_PREDS}.retention_actions`
    SELECT
      s.snapshot_month,
      s.employee_code,
      s.country,
      s.team,
      s.position,
      `{PROJECT_ID}.{DS_GOLD}.tier_from_position`(s.position) AS tier,
      s.prob_class_1 AS retention_risk_score,
      cp.tau_star,
      CASE
        WHEN s.prob_class_1 >= cp.tau_star THEN 'retention_action'
        WHEN s.prob_class_1 >= cp.tau_star * 0.85 THEN 'monitor'
        ELSE 'no_action'
      END AS recommended_action,
      CASE
        WHEN s.prob_class_1 >= 0.95 THEN 1
        WHEN s.prob_class_1 >= cp.tau_star THEN 2
        WHEN s.prob_class_1 >= cp.tau_star * 0.85 THEN 3
        WHEN s.prob_class_1 >= 0.5 THEN 4
        ELSE 5
      END AS priority,
      cp.Cu_pct AS replacement_cost_pct,
      cp.Co_pct AS retention_cost_pct,
      CURRENT_TIMESTAMP() AS scored_at
    FROM `{PROJECT_ID}.{DS_PREDS}.scores` s
    JOIN `{PROJECT_ID}.{DS_GOLD}.cost_params_retention` cp
      ON cp.tier = `{PROJECT_ID}.{DS_GOLD}.tier_from_position`(s.position)
    WHERE s.snapshot_month = target_month;

    SET rows_written = @@row_count;

    INSERT INTO `{PROJECT_ID}.{DS_RUNS}.retention_pipeline_runs`
      (run_id, step, snapshot_month, rows_written, start_time, end_time, duration_seconds, status)
    VALUES
      (run_uuid, 'apply_decision', target_month, rows_written, start_ts, CURRENT_TIMESTAMP(),
       TIMESTAMP_DIFF(CURRENT_TIMESTAMP(), start_ts, MILLISECOND) / 1000.0, 'SUCCESS');

  EXCEPTION WHEN ERROR THEN
    INSERT INTO `{PROJECT_ID}.{DS_RUNS}.retention_pipeline_runs`
      (run_id, step, snapshot_month, rows_written, start_time, end_time, status, error_message)
    VALUES
      (run_uuid, 'apply_decision', target_month, 0, start_ts, CURRENT_TIMESTAMP(), 'FAILED', @@error.message);
    RAISE USING MESSAGE = @@error.message;
  END;
END
'''

# Antes del SP, crear la tabla de actions
bq_client.query(f'''
CREATE TABLE IF NOT EXISTS `{PROJECT_ID}.{DS_PREDS}.retention_actions` (
  snapshot_month DATE NOT NULL,
  employee_code INT64 NOT NULL,
  country STRING,
  team STRING,
  position STRING,
  tier STRING,
  retention_risk_score FLOAT64,
  tau_star FLOAT64,
  recommended_action STRING,
  priority INT64,
  replacement_cost_pct FLOAT64,
  retention_cost_pct FLOAT64,
  scored_at TIMESTAMP
)
PARTITION BY snapshot_month
CLUSTER BY country, recommended_action
OPTIONS(
  description="Lista priorizada de acciones de retención. Una fila por (empleado×snapshot). Output final del pipeline.",
  labels = [("team", "people-analytics"), ("pipeline", "retention-risk"), ("contains_pii", "false")]
)
''').result()

bq_client.query(sp_apply_decision).result()
print(f"SP creado: {DS_PREDS}.sp_apply_decision_rule")

# Ejecutar para los 3 meses de test
for m in TEST_MONTHS:
    bq_client.query(
        f"CALL `{PROJECT_ID}.{DS_PREDS}.sp_apply_decision_rule`(DATE '{m}')",
        job_config=bigquery.QueryJobConfig(labels={**LABELS_PIPELINE, "step": "apply_decision"})
    ).result()
    print(f"  Decisiones aplicadas para {m}")


In [ ]:
# Distribución de acciones recomendadas
df_actions = bq_client.query(f'''
SELECT
  snapshot_month,
  recommended_action,
  COUNT(*) AS empleados,
  ROUND(AVG(retention_risk_score), 3) AS avg_score,
  ROUND(MIN(retention_risk_score), 3) AS min_score,
  ROUND(MAX(retention_risk_score), 3) AS max_score
FROM `{PROJECT_ID}.{DS_PREDS}.retention_actions`
WHERE snapshot_month IN UNNEST([{', '.join(f"DATE '{m}'" for m in TEST_MONTHS)}])
GROUP BY snapshot_month, recommended_action
ORDER BY snapshot_month, recommended_action
''').to_dataframe()
print("Distribución de acciones recomendadas:")
print(df_actions.to_string(index=False))
print()

# Lista priorizada para 2025-11 (último mes test)
df_top = bq_client.query(f'''
SELECT employee_code, country, team, position, tier, recommended_action, priority, ROUND(retention_risk_score,3) AS score
FROM `{PROJECT_ID}.{DS_PREDS}.retention_actions`
WHERE snapshot_month = DATE '2025-11-01'
  AND recommended_action IN ('retention_action', 'monitor')
ORDER BY priority, score DESC
''').to_dataframe()
print(f"Lista para HRBP en 2025-11: {len(df_top)} empleados sobre los que actuar/monitorizar")
print(df_top.head(15).to_string(index=False))


---
## 11. Comparison plot — modelo vs naive base rate

Espejo de la sección 10 del template: comparamos el coste esperado del **modelo entrenado** contra una **regla naive** (asignar a todo el mundo el mismo score = base rate). Si nuestro modelo no supera al naive, **no merece la pena ponerlo en producción**.

In [ ]:
# Calcular coste esperado total bajo dos reglas, para todo el test set
df_test = bq_client.query(f'''
SELECT
  s.employee_code,
  s.snapshot_month,
  s.actual_label,
  s.prob_class_1 AS model_score,
  cp.tau_star,
  cp.Cu_pct,
  cp.Co_pct
FROM `{PROJECT_ID}.{DS_PREDS}.scores_test` s
JOIN `{PROJECT_ID}.{DS_GOLD}.cost_params_retention` cp
  ON cp.tier = `{PROJECT_ID}.{DS_GOLD}.tier_from_position`(s.position)
''').to_dataframe()

# Asumimos un salario medio para cuantificar (en producción usaríamos salario individual)
SALARY_MEAN = 60000  # EUR

# Coste esperado por threshold (modelo)
thresholds = np.linspace(0.05, 0.99, 50)
results = []
for tau in thresholds:
    df_test['action_model'] = (df_test['model_score'] >= tau).astype(int)
    cost_action = df_test['action_model'] * df_test['Co_pct'] * SALARY_MEAN
    cost_no_action_miss = (1 - df_test['action_model']) * df_test['actual_label'] * df_test['Cu_pct'] * SALARY_MEAN
    total_cost = (cost_action + cost_no_action_miss).sum()
    n_actions = df_test['action_model'].sum()
    results.append({"tau": tau, "total_cost": total_cost, "n_actions": n_actions})

df_curve = pd.DataFrame(results)

# Coste de la regla naive: aplicar acción a TODO el mundo (tau=0) o A NADIE (tau=1)
base_rate = df_test['actual_label'].mean()
n_total = len(df_test)
cost_act_all = n_total * df_test['Co_pct'].mean() * SALARY_MEAN
cost_act_none = df_test['actual_label'].sum() * df_test['Cu_pct'].mean() * SALARY_MEAN

# Plot comparativo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df_curve['tau'], df_curve['total_cost'] / 1000, label='Modelo logistic', linewidth=2)
axes[0].axhline(cost_act_all / 1000, color='red', linestyle='--', label='Naive: actuar sobre todos')
axes[0].axhline(cost_act_none / 1000, color='orange', linestyle='--', label='Naive: no actuar sobre nadie')
axes[0].axvline(df_curve.loc[df_curve['total_cost'].idxmin(), 'tau'], color='green', linestyle=':', label=f"τ óptimo modelo")
axes[0].set_xlabel('Threshold τ')
axes[0].set_ylabel('Coste esperado total (k€)')
axes[0].set_title('Coste esperado vs threshold')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(df_curve['tau'], df_curve['n_actions'], color='purple', linewidth=2)
axes[1].set_xlabel('Threshold τ')
axes[1].set_ylabel('# Empleados con acción de retención')
axes[1].set_title('Volumen de intervención HRBP')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/retention_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

# Lectura honesta
opt = df_curve.loc[df_curve['total_cost'].idxmin()]
print()
print(f"Base rate de voluntary_exit en test: {base_rate:.1%}")
print(f"Coste mínimo modelo:    {opt['total_cost']/1000:>10.1f} k€  (τ={opt['tau']:.2f}, {opt['n_actions']:.0f} acciones)")
print(f"Coste 'actuar sobre todos':  {cost_act_all/1000:>10.1f} k€")
print(f"Coste 'no actuar sobre nadie': {cost_act_none/1000:>10.1f} k€")
mejora = (min(cost_act_all, cost_act_none) - opt['total_cost']) / min(cost_act_all, cost_act_none) * 100
print(f"Mejora del modelo sobre la mejor regla naive: {mejora:.1f}%")


---
## 12. Capa semántica + authorized views + snapshots (T6.5, T6.6, T6.7, T6.8)

Tres patrones de buenas prácticas en una sección:

1. **Vista semántica** `v_retention_risk_dashboard` — la única fuente de verdad para los dashboards.
2. **Authorized view** — los HRBP pueden leer la vista sin tener acceso directo a la tabla base con scores.
3. **Snapshots** — cada ejecución del pipeline genera un snapshot inmutable para auditoría posterior.

In [ ]:
# 1. Vista semántica para Looker Studio / Connected Sheets
view_sql = f'''
CREATE OR REPLACE VIEW `{PROJECT_ID}.{DS_GOLD}.v_retention_risk_dashboard` AS
SELECT
  ra.snapshot_month,
  ra.country,
  ra.team,
  ra.tier,
  ra.recommended_action,
  ra.priority,
  COUNT(*) AS empleados,
  ROUND(AVG(ra.retention_risk_score), 3) AS avg_score,
  ROUND(AVG(ra.tau_star), 3) AS avg_threshold
FROM `{PROJECT_ID}.{DS_PREDS}.retention_actions` ra
GROUP BY snapshot_month, country, team, tier, recommended_action, priority
HAVING COUNT(*) >= 5   -- k-anonymity: nunca cohortes de menos de 5
'''

bq_client.query(view_sql).result()

bq_client.query(f'''
ALTER VIEW `{PROJECT_ID}.{DS_GOLD}.v_retention_risk_dashboard`
SET OPTIONS(
  description="Vista semántica oficial para dashboards de retention risk. Agrega por (mes, país, banda, acción). k-anonymity ≥5."
)
''').result()
print(f"Vista creada: {DS_GOLD}.v_retention_risk_dashboard")


In [ ]:
# 2. Authorized view — la vista puede leer la tabla base aunque el usuario no tenga acceso directo
view_ref = bq_client.get_table(f"{PROJECT_ID}.{DS_GOLD}.v_retention_risk_dashboard").reference

# Autorizar la vista para acceder al dataset de predictions
ds_preds = bq_client.get_dataset(f"{PROJECT_ID}.{DS_PREDS}")
new_entry = bigquery.AccessEntry(
    role=None,
    entity_type="view",
    entity_id={
        "projectId": PROJECT_ID,
        "datasetId": DS_GOLD,
        "tableId": "v_retention_risk_dashboard",
    },
)
existing = list(ds_preds.access_entries)
if not any(e.entity_id == new_entry.entity_id for e in existing if hasattr(e, 'entity_id') and isinstance(e.entity_id, dict)):
    existing.append(new_entry)
    ds_preds.access_entries = existing
    bq_client.update_dataset(ds_preds, ["access_entries"])
    print(f"Authorized view: {DS_GOLD}.v_retention_risk_dashboard puede leer {DS_PREDS}.*")
else:
    print(f"Authorized view ya configurada")

print()
print("Patrón aplicado: HRBP puede consultar v_retention_risk_dashboard sin necesidad de acceso a")
print(f"{DS_PREDS}.retention_actions (que contiene scores individuales).")


In [ ]:
# 3. Snapshots de auditoría — uno por mes de test
# Patrón: snapshot inmutable con expiration. Si descubrimos un bug en el modelo dentro de 6 meses,
# podemos auditar exactamente qué scores dimos.

today = date.today().strftime('%Y%m%d')
for m in TEST_MONTHS:
    m_clean = m.replace('-', '')
    snapshot_name = f"retention_actions_snapshot_{m_clean}_{today}"
    sql = f'''
    CREATE SNAPSHOT TABLE `{PROJECT_ID}.{DS_PREDS}.{snapshot_name}`
    CLONE `{PROJECT_ID}.{DS_PREDS}.retention_actions`
    OPTIONS(
      expiration_timestamp = TIMESTAMP_ADD(CURRENT_TIMESTAMP(), INTERVAL 365 DAY),
      description = "Snapshot de auditoría — decisiones del pipeline para snapshot_month {m}"
    )
    '''
    try:
        bq_client.query(sql).result()
        print(f"  Snapshot creado: {snapshot_name}")
    except Conflict:
        print(f"  Snapshot ya existe: {snapshot_name}")

print()
print("Snapshots tienen expiración de 1 año. Coste de almacenamiento: ~0 mientras la base no diverja.")


---
## 13. INFORMATION_SCHEMA cost audit (T6.10)

Auditoría de coste y rendimiento del pipeline. Esta query la corremos **después** de cada ejecución para responder a:
- ¿Cuánto costó este pipeline run?
- ¿Qué step es el cuello de botella?
- ¿Hay queries que estamos repitiendo innecesariamente?

En M12 (FinOps) profundizaremos esta vista hasta tener un dashboard completo.

In [ ]:
# Auditoría de coste y rendimiento de las queries del pipeline en las últimas 24h
# Filtramos por label pipeline=retention-risk para aislar nuestras queries
# INFORMATION_SCHEMA en BQ se accede via region-<prefix>: europe-southwest1 → region-europe-southwest1
INFO_SCHEMA_REGION = f"region-{REGION}"

audit_sql = f'''
SELECT
  step_label.value AS step,
  COUNT(*) AS jobs,
  ROUND(SUM(total_bytes_processed) / POW(1024, 3), 3) AS gb_processed,
  ROUND(SUM(total_bytes_billed) / POW(1024, 3), 3) AS gb_billed,
  ROUND(SUM(total_bytes_billed) / POW(1024, 4) * 5, 4) AS estimated_cost_usd,
  ROUND(SUM(total_slot_ms) / 1000.0, 1) AS slot_seconds,
  ROUND(AVG(TIMESTAMP_DIFF(end_time, start_time, MILLISECOND))/1000.0, 2) AS avg_duration_s
FROM `{INFO_SCHEMA_REGION}`.INFORMATION_SCHEMA.JOBS_BY_USER j,
  UNNEST(j.labels) AS step_label
WHERE j.creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR)
  AND j.job_type = 'QUERY'
  AND j.state = 'DONE'
  AND step_label.key = 'step'
  AND EXISTS (
    SELECT 1 FROM UNNEST(j.labels) AS l
    WHERE l.key = 'pipeline' AND l.value = 'retention-risk'
  )
GROUP BY step
ORDER BY estimated_cost_usd DESC
'''

# Nota: usamos JOBS_BY_USER (solo nuestros jobs) en lugar de JOBS_BY_PROJECT.
# JOBS_BY_PROJECT requiere roles/bigquery.resourceViewer; JOBS_BY_USER funciona con roles/bigquery.user.
# En producción con un Service Account dedicado al pipeline, JOBS_BY_PROJECT sería preferible.
try:
    df_audit = bq_client.query(audit_sql).to_dataframe()
    if len(df_audit) > 0:
        print("Auditoría FinOps del pipeline (últimas 24h, jobs de este usuario):")
        print(df_audit.to_string(index=False))
        print()
        print(f"Coste total estimado:   ${df_audit['estimated_cost_usd'].sum():.4f}")
        print(f"Bytes procesados total: {df_audit['gb_processed'].sum():.2f} GiB")
    else:
        print("No hay queries del pipeline en INFORMATION_SCHEMA aún (puede tardar 1-2 min en aparecer).")
except Exception as e:
    print(f"INFORMATION_SCHEMA no accesible aún: {type(e).__name__}: {str(e)[:300]}")
    print("Si es 403 Forbidden, asigna roles/bigquery.resourceViewer al usuario para JOBS_BY_PROJECT.")


In [ ]:
# Vista combinada: pipeline_runs + INFORMATION_SCHEMA → 'estado del pipeline'
state_sql = f'''
SELECT
  step,
  COUNT(*) AS ejecuciones,
  COUNTIF(status = 'SUCCESS') AS exitos,
  COUNTIF(status = 'FAILED') AS fallos,
  ROUND(AVG(duration_seconds), 2) AS avg_duration_s,
  ROUND(SUM(rows_written), 0) AS total_rows_written
FROM `{PROJECT_ID}.{DS_RUNS}.retention_pipeline_runs`
WHERE DATE(start_time) >= CURRENT_DATE() - 1
GROUP BY step
ORDER BY ejecuciones DESC
'''
df_state = bq_client.query(state_sql).to_dataframe()
print("Estado del pipeline (pipeline_runs):")
print(df_state.to_string(index=False))


---
## 14. GenAI brief stub (foreshadowing M16)

El template *procurement* tenía una sección "GenAI procurement brief". Nosotros hacemos el equivalente para retention: una función que toma la lista priorizada y genera un **brief en lenguaje natural** para el HRBP.

Aquí solo dejamos el **stub** — la implementación completa con Vertex AI Gemini llega en M16.

In [ ]:
def build_retention_brief(snapshot_month: str, top_n: int = 5) -> str:
    """Genera un brief de retención en lenguaje natural a partir de las decisiones del pipeline.

    Stub pedagógico — la implementación real con Vertex AI Gemini se hará en Módulo 16.
    Aquí solo construimos el contexto que se pasaría al modelo.
    """
    df = bq_client.query(f"""
    SELECT employee_code, country, team, tier, recommended_action, priority,
           ROUND(retention_risk_score, 3) AS score
    FROM `{PROJECT_ID}.{DS_PREDS}.retention_actions`
    WHERE snapshot_month = DATE '{snapshot_month}'
      AND recommended_action IN ('retention_action', 'monitor')
    ORDER BY priority, score DESC
    LIMIT {top_n}
    """).to_dataframe()

    contexto = {
        "snapshot_month": snapshot_month,
        "n_acciones": len(df),
        "casos": df.to_dict("records"),
    }

    # === En M16 sustituiremos esto por una llamada a Vertex AI Gemini ===
    # from vertexai.generative_models import GenerativeModel
    # model = GenerativeModel("gemini-1.5-pro")
    # prompt = f"Eres un HRBP. Redacta un brief para el manager basado en: {contexto}"
    # response = model.generate_content(prompt)
    # return response.text

    # Por ahora, plantilla simple en Python
    lineas = []
    lineas.append(f"BRIEF DE RETENCIÓN — Mes {snapshot_month}")
    lineas.append("=" * 50)
    lineas.append("")
    lineas.append(f"Se han identificado {len(df)} casos prioritarios:")
    lineas.append("")
    for _, row in df.iterrows():
        emp = str(row["employee_code"])
        lineas.append(f"  - Empleado #{emp} ({row['country']}, team {row['team']}, tier {row['tier']}):")
        lineas.append(f"      Score = {row['score']}, prioridad {row['priority']}, acción: {row['recommended_action']}")
    lineas.append("")
    lineas.append("Recomendación: programar 1:1 con el manager de cada caso de prioridad 1-2 en las próximas 2 semanas.")
    lineas.append("")
    lineas.append("[En M16 esto será generado por Vertex AI Gemini con prompts contextualizados]")
    return "\n".join(lineas)


print(build_retention_brief("2025-11-01"))


---
## 15. Vertex AI Model Registry wiring (foreshadowing M15)

Espejo de la sección 12 del template procurement. Cuando promocionemos este modelo a producción "real" (M15):

- Lo registraremos en **Vertex AI Model Registry** con versión, métricas, lineage.
- Crearemos un **Endpoint** para servir predicciones online si hace falta latencia <100ms.
- Habilitaremos **Model Monitoring** para detectar drift de features y degradación de métricas.

Aquí dejamos el código **comentado** — se activa en M15.

In [ ]:
# === Vertex AI Model Registry — activado en M15 ===
# from google.cloud import aiplatform
#
# aiplatform.init(project=PROJECT_ID, location=REGION)
#
# # 1. Registrar el modelo BQML en Vertex AI Model Registry
# # BQML models pueden exportarse o referenciarse directamente
# model = aiplatform.Model.upload(
#     display_name="retention-logistic",
#     description="BQML logistic regression — retention risk monthly",
#     artifact_uri=f"bq://{PROJECT_ID}.{DS_MODELS}.retention_logistic_v1",
#     serving_container_image_uri="europe-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest",
#     labels={**LABELS_PIPELINE, "model_type": "logistic"},
# )
#
# # 2. Endpoint para predicciones online (opcional — si HRBP necesita scoring on-demand)
# endpoint = model.deploy(
#     machine_type="n1-standard-2",
#     min_replica_count=1,
#     max_replica_count=2,
#     traffic_split={"0": 100},
# )
#
# # 3. Model Monitoring — alerta si se degrada AUC o si las features hacen drift
# from google.cloud.aiplatform.model_monitoring import ModelMonitoringJob
# monitoring_job = ModelMonitoringJob.create(
#     display_name="retention-monitoring",
#     model_id=model.resource_name,
#     monitoring_interval_hours=24,
#     alert_emails=["data-eng-team@imagina.es"],
# )
#
# print(f"Modelo registrado: {model.resource_name}")
# print(f"Endpoint:          {endpoint.resource_name}")
# === / Vertex AI ===

print("Bloque comentado — se activa en Módulo 15 (Vertex AI Aplicado a Datos de Personas).")


---
## 16. Findings — honest read

Espejo directo de la sección 13 del template procurement. **Esta es la sección más importante del notebook**: si no eres honesto sobre las limitaciones, terminas desplegando un modelo que toma decisiones sobre personas con datos insuficientes.

### Lo que funcionó

1. **El pipeline corre end-to-end** y es idempotente: `sp_backfill_features('2025-01-01', '2025-12-01')` se puede re-ejecutar sin duplicar filas ni romper logs.
2. **La regla de decisión newsvendor** es defendible ante negocio: cada acción tiene un coste comparado contra un coste esperado de no-acción.
3. **La observabilidad por defecto** (pipeline_runs + INFORMATION_SCHEMA) nos permite responder a "¿qué pasó el día X?" en segundos.
4. **Particionado y require_partition_filter** evitan queries asesinas — un analista que haga `SELECT * FROM features` recibe error en lugar de leer 24 meses de datos.

### Lo que NO funcionó / hay que tener cuidado

1. **El dataset es minúsculo**: 209 empleados × 12 meses = ~2,500 filas de entrenamiento. Cualquier métrica de validación tiene **incertidumbre enorme**. AUC=0.72 podría perfectamente ser AUC=0.55 con otro split — el intervalo de confianza es del tamaño del rango entero.

2. **Tasa de exits del 41%** es sospechosamente alta. O la empresa pasó por un downsizing (en cuyo caso el patrón aprendido **no generaliza** a un mes normal), o el export incluye empleados históricos que ya no están (lo que rompe el supuesto de "snapshot al inicio del mes"). Pendiente: validar con cliente antes de poner en prod.

3. **El label es ruidoso**: `voluntary_exit_within_3m` se basa en `exit_type` que puede estar mal codificado (alguien que se "fue voluntariamente" para no ser despedido aparece como Voluntary). En PA esto es la regla, no la excepción.

4. **Variables omitidas críticas**: no tenemos satisfacción/engagement, performance review, manager NPS, jornada nocturna, viajes recientes. Lo que tenemos (compa-ratio, tenure, salary delta) explica una fracción pequeña del riesgo real de salida.

5. **Sesgo potencial — bandera roja**: el modelo podría aprender a discriminar por `country` si las prácticas de offboarding difieren entre países (algunas oficinas dan más bajas voluntarias que otras por cultura local, no por riesgo individual). En M11 (DLP + governance) habrá que añadir tests de fairness por subgrupo.

6. **Nunca usar este score solo**: la regla GDPR Art. 22 (decisiones automatizadas) prohíbe que una predicción afecte significativamente a un trabajador sin revisión humana. **El output del pipeline es input para HRBP, no decisión final.**

### Próximos pasos (módulos siguientes)

- **M5** (siguiente bloque de la sesión): orquestar este pipeline con Cloud Workflows + Scheduler para que corra automáticamente cada día 1 del mes.
- **M7** (SQL avanzado): añadir window functions de cohorte para mejorar features (turnover de cohorte, comparativas peer-group).
- **M8** (Dataform): versionar todos estos SPs y UDFs en Git con CI/CD.
- **M11** (DLP + governance): aplicar Cloud DLP a las features, tests de fairness por subgrupo.
- **M12** (FinOps): el dashboard que vimos en `INFORMATION_SCHEMA` se conecta a BI Engine.
- **M14** (Looker Studio): dashboard de HRBP construido sobre `v_retention_risk_dashboard`.
- **M15** (Vertex AI): sustituir el BQML logistic por modelos más sofisticados (AutoML Tabular, XGBoost) y desplegar en endpoint.
- **M16** (GenAI): convertir el stub `build_retention_brief` en una llamada real a Gemini.

---

## Resumen del módulo

Este notebook ha sido un viaje completo:

1. ✅ Modelado de datasets de producción (T6.1)
2. ✅ Particionamiento y clustering con `require_partition_filter` (T6.2)
3. ✅ Control de coste con `maximum_bytes_billed` y labels (T6.3)
4. ✅ Optimización (filtros antes de JOINs, predicate pushdown) (T6.4)
5. ✅ Authorized views con k-anonymity (T6.5)
6. ✅ Capa semántica `v_retention_risk_dashboard` (T6.6)
7. ✅ Versionado implícito (model `_v1`) (T6.7)
8. ✅ Snapshots de auditoría (T6.8)
9. ✅ Documentación con `OPTIONS(description=...)` (T6.9)
10. ✅ Auditoría con `INFORMATION_SCHEMA.JOBS_BY_PROJECT` + tabla `pipeline_runs` (T6.10)
11. ✅ Stored Procedures, UDFs y Procedural Language (T6.11)

Y todo aplicado a un **caso real de People Analytics** con datos reales anonimizados, una regla de decisión defendible y findings honestos sobre lo que el modelo puede y no puede hacer.

→ **Continuamos en el Módulo 5: orquestar este pipeline con Cloud Workflows y Cloud Composer.**
